# OpenAPI를 활용할 네이버 뉴스 검색

## 1. 애플리케이션 등록
https://developers.naver.com/apps/#/register

## 2. 환경 변수 관리
- 등록된 애플리케이션에서 제공되는 Client ID, Secret은 외부 노출 금지
- dotenv (.env)를 통해서 관리
    - `pip install dotenv` 설치 필요
    - .gitignore 파일에 .env 무시하는 구문 추가



In [18]:
import json
from asyncio import timeout

from IPython.core.magics import display
!pip install python-dotenv

### project 하위 폴더에 .env 파일을 만들어 아래 값을 입력
```
NAVER_CLIENT_ID = {{Client Id}}
NAVER_CLINET_SECRET = {{Client Secret}}
```

- 네이버 개발자 센터에서 확인 가능

In [19]:
# .env 파일을 로드해서 환경변수로 등록
from dotenv import load_dotenv

load_dotenv() # 같은 경로상의 .env를 읽어와 자동으로 환경변수로 등록

# 읽어 오기 성공 여부에 따라 bool 반환

True

In [20]:
# 환경 변수에서 .env 등록 내역 업어오기
import os

NAVER_CLIENT_ID = os.getenv("NAVER_CLIENT_ID")
NAVER_CLIENT_SECRET = os.getenv("NAVER_CLIENT_SECRET")

# api key, 비밀번호를 print 하는 구문 실수로 남길 경우 유출 가능성 있기에
# -> 주피터 변수 탬에서 확인

In [21]:
# .env 내영이 환경 변수로 등록 되지 않은 경우
if not NAVER_CLIENT_ID or not NAVER_CLIENT_SECRET:
    raise ValueError("NAVER_CLIENT_ID or NAVER_CLIENT_SECRET 이 환경 변수에 등록 안됨")

## 3. API 요청
- 파이썬에서 웹 요청(http)을 처리하기 위해선 `requests` 라이브러리 필요

```
!pip install requests
```

In [22]:
# 네이버 뉴스 검색 API 요청
import urllib.request
import socket

encText = urllib.parse.quote('인공지능') # url encoding 작업
# 한글 -> %시작하는 문자로 변경

url = f'https://openapi.naver.com/v1/search/news.json?query={encText}&display=10&sort=date'

# 요청 객체 생성
request = urllib.request.Request(url)

# API 인증 정보를 요청 헤더에 추가
request.add_header("X-Naver-Client-Id", NAVER_CLIENT_ID)
request.add_header("X-Naver-Client-Secret", NAVER_CLIENT_SECRET)

try:
    with urllib.request.urlopen(request, timeout=10) as response:
        # 지정된 주소로 요청 -> 결과를 response로 전달 받음
        # 단, 요청 대기시간이 10초를 초과하면 중지

        # HTTP 응답 상태 코드. 200이면 정상 응답 == 성공
        response_code = response.getcode()

        # 응답 본문 확인 (bytes -> UTF-8로 변환)
        response_body = response.read().decode('utf-8')

        print("response_code :", response_code)
        print(response_body)
        # 응답 본문이 JSON(str type)형태 -> 이용을 위해 파싱 작업필수

except socket.timeout:
    print("요청 시간 10초 초과")


response_code : 200
{
	"lastBuildDate":"Mon, 15 Jun 2026 17:26:01 +0900",
	"total":4047408,
	"start":1,
	"display":10,
	"items":[
		{
			"title":"스페이스X 로켓 뜨면 삼성·SK하이닉스 반도체도 '난다'",
			"originallink":"https:\/\/www.sisaweek.com\/news\/articleView.html?idxno=236122",
			"link":"https:\/\/www.sisaweek.com\/news\/articleView.html?idxno=236122",
			"description":"또한 최근 도입되는 '<b>인공지능<\/b>(AI)' 기반 우주데이터센터도 마찬가지다. 따라서 우주 기반 AI인프라가 확대될수록 고성능 연산칩과 메모리 반도체의 중요성은 더욱 커질 수밖에 없다. 실제로 관련 시장 규모는 매해... ",
			"pubDate":"Mon, 15 Jun 2026 17:24:00 +0900"
		},
		{
			"title":"한컴, 유럽 AI·R&amp;D 기업과 협력 강화…에이전틱 OS 현지화 추진",
			"originallink":"https:\/\/daily.hankooki.com\/news\/articleView.html?idxno=1376737",
			"link":"https:\/\/daily.hankooki.com\/news\/articleView.html?idxno=1376737",
			"description":"사진=한컴 제공  한컴이 유럽 현지 <b>인공지능<\/b>(AI) 및 연구개발(R&amp;D) 기업들과 업무협약(MOU)을 체결하고 제품 공동 개발과 현지 시장 진출 기반 확보에 나선다. 한컴은 폴란드 국가공인 연구개발(R&amp;D) 센터 '7불스'와 MOU를... ",
			"pubDate":"Mon, 15 Jun 2026 17:24:00 +0900"
		},
		{


In [23]:
# requests 객체를 이용한 요청(더 쉬움)
import requests
from pprint import pprint # 출력시 엔터

url = 'https://openapi.naver.com/v1/search/news.json'

# Header, Body에 전달할 값을 dict 형식으로 생성
headers = {
    "X-Naver-Client-id": NAVER_CLIENT_ID,
    "X-Naver-Client-Secret": NAVER_CLIENT_SECRET
}

params = {
    'query' : '인공지능',
    'display' : 10,
    'start' : 1,
    'sort' : 'date'
}

try:
    # GET Method == 조회 요청
    response = requests.get(
        url,
        headers=headers,
        params=params, # dict -> 쿼리 스트링 변환(+url encoding)
        timeout=10
    )

    # HTTP 상태 코드가 400, 500번대 인 경우 예외 발생
    response.raise_for_status()

    response_code = response.status_code # 상태 코드
    data = response.json() # 응답 데이터(json) -> dict 변환

    print('response_code: ', response_code)
    # pprint(data)
    pprint(data['items'][0])
except requests.exceptions.Timeout:
    print("요청 시간 10초 초과")
except ValueError:
    print("응답 데이터가 JSON 형식이 아닙니다")

response_code:  200
{'description': "또한 최근 도입되는 '<b>인공지능</b>(AI)' 기반 우주데이터센터도 마찬가지다. 따라서 우주 기반 "
                'AI인프라가 확대될수록 고성능 연산칩과 메모리 반도체의 중요성은 더욱 커질 수밖에 없다. 실제로 관련 시장 '
                '규모는 매해... ',
 'link': 'https://www.sisaweek.com/news/articleView.html?idxno=236122',
 'originallink': 'https://www.sisaweek.com/news/articleView.html?idxno=236122',
 'pubDate': 'Mon, 15 Jun 2026 17:24:00 +0900',
 'title': "스페이스X 로켓 뜨면 삼성·SK하이닉스 반도체도 '난다'"}


In [24]:
# requests 객체를 이용한 요청(더 쉬움)
import requests
from pprint import pprint # 출력시 엔터

url = 'https://openapi.naver.com/v1/search/news.json'

# Header, Body에 전달할 값을 dict 형식으로 생성
headers = {
    "X-Naver-Client-id": NAVER_CLIENT_ID,
    "X-Naver-Client-Secret": NAVER_CLIENT_SECRET
}

params = {
    'query' : '인공지능',
    'display' : 10,
    'start' : 1,
    'sort' : 'date'
}

try:
    # GET Method == 조회 요청
    response = requests.get(
        url,
        headers=headers,
        params=params, # dict -> 쿼리 스트링 변환(+url encoding)
        timeout=10
    )

    # HTTP 상태 코드가 400, 500번대 인 경우 예외 발생
    response.raise_for_status()

    response_code = response.status_code # 상태 코드
    data = response.content # 응답 데이터(json) -> dict 변환

    #xml 해석 내장 모듈 -> xml.etree.ElementTree

    with open('news.xml', 'wb') as f:
        f.write(data)

    print('newsss.xml 저장 완료')

except requests.exceptions.Timeout:
    print("요청 시간 10초 초과")

newsss.xml 저장 완료
